Insert from Files to Dataframes

In [1]:
import pandas as pd
import os
from IPython.display import display


# Ο κεντρικός φάκελος με τα δεδομένα (βρίσκεται στον ίδιο φάκελο με το .ipynb)
base_dir = "../dataset-understat"
leagues = ["Bundesliga", "EPL", "La_Liga", "Ligue_1", "RFPL", "Serie_A"]

df_list = []

# 1. Διάβασμα όλων των αρχείων shot_data.csv
for league in leagues:
    file_path = os.path.join(base_dir, league, "shot_data.csv")
    if os.path.exists(file_path):
        df_temp = pd.read_csv(file_path)
        df_list.append(df_temp)
    else:
        print(f"Προειδοποίηση: Δεν βρέθηκε το αρχείο {file_path}")

# 2. Συνένωση σε ένα κεντρικό DataFrame
if df_list:
    df_raw_shots = pd.concat(df_list, ignore_index=True)

    # 3. Δημιουργία του τελικού DataFrame με βάση το σχήμα της Βάσης Δεδομένων (Table: understat_shots)
    df_understat_shots = pd.DataFrame()

    # --- ΑΝΑΓΝΩΡΙΣΤΙΚΑ ---
    df_understat_shots['shot_id'] = df_raw_shots['id']
    df_understat_shots['match_id'] = df_raw_shots['match_id']
    df_understat_shots['uid_understat'] = df_raw_shots['player_id']

    # --- ΧΡΟΝΟΣ & CONTEXT ---
    df_understat_shots['minute'] = df_raw_shots['minute']
    df_understat_shots['season'] = df_raw_shots['season']
    df_understat_shots['league'] = df_raw_shots['league']
    df_understat_shots['h_a'] = df_raw_shots['h_a']

    # --- ΓΕΩΜΕΤΡΙΑ ---
    df_understat_shots['x_loc'] = df_raw_shots['X']
    df_understat_shots['y_loc'] = df_raw_shots['Y']
    # Τα distance_to_goal & angle_to_goal δημιουργούνται αυτόματα στη ΒΔ (GENERATED ALWAYS), οπότε δεν τα βάζουμε.

    # --- ΔΗΜΙΟΥΡΓΙΑ ΦΑΣΗΣ ---
    df_understat_shots['play_pattern'] = df_raw_shots['situation']
    df_understat_shots['last_action'] = df_raw_shots['lastAction']
    df_understat_shots['player_assisted'] = df_raw_shots['player_assisted']

    # Σημείωση: Το uid_assisted θέλει join με το player.csv. Για τώρα το αφήνουμε κενό (NaN)
    # μέχρι να μου δώσεις το επόμενο βήμα.
    df_understat_shots['uid_assisted'] = pd.NA

    # --- ΜΗΧΑΝΙΚΗ ΣΟΥΤ ---
    df_understat_shots['body_part'] = df_raw_shots['shotType']

    # --- ΑΠΟΤΕΛΕΣΜΑ & TARGETS ---
    df_understat_shots['outcome'] = df_raw_shots['result']
    # Derived logic για το is_goal
    df_understat_shots['is_goal'] = df_raw_shots['result'].apply(lambda x: 1 if x == 'Goal' else 0)
    df_understat_shots['understat_xg'] = df_raw_shots['xG']

    print("Το DataFrame df_understat_shots δημιουργήθηκε επιτυχώς!")
    print(f"Συνολικό πλήθος σουτ: {len(df_understat_shots)}")

    # Εμφάνιση των πρώτων 5 εγγραφών
    display(df_understat_shots.head())
else:
    print("Σφάλμα: Δεν φορτώθηκαν καθόλου δεδομένα.")

Το DataFrame df_understat_shots δημιουργήθηκε επιτυχώς!
Συνολικό πλήθος σουτ: 615225


,shot_id,match_id,uid_understat,minute,season,league,h_a,x_loc,y_loc,play_pattern,last_action,player_assisted,uid_assisted,body_part,outcome,is_goal,understat_xg
0,24339,5447,224,11,2014,Bundesliga,h,0.903,0.239,OpenPlay,Chipped,Philipp Lahm,<NA>,RightFoot,SavedShot,0,0.034118
1,24340,5447,392,17,2014,Bundesliga,h,0.852,0.277,OpenPlay,Pass,Philipp Lahm,<NA>,LeftFoot,SavedShot,0,0.030941
2,24342,5447,392,26,2014,Bundesliga,h,0.803,0.277,OpenPlay,Chipped,Holger Badstuber,<NA>,LeftFoot,BlockedShot,0,0.021718
3,24343,5447,224,28,2014,Bundesliga,h,0.871,0.324,OpenPlay,NaN,NaN,<NA>,LeftFoot,SavedShot,0,0.050345
4,24344,5447,227,29,2014,Bundesliga,h,0.918,0.531,OpenPlay,Chipped,Gianluca Gaudino,<NA>,RightFoot,SavedShot,0,0.111078


Ευρεση id του player που εκανε την assist (εαν υπαρχει assist και δεν ειναι NaN)

In [21]:
import html

def clean_name(raw_name):
    if pd.isna(raw_name) or raw_name == '':
        return raw_name
    return html.unescape(str(raw_name)).strip()

# 1. Διάβασμα των player.csv
df_players_list = []
for league in leagues:
    player_file = os.path.join(base_dir, league, "player.csv")
    if os.path.exists(player_file):
        df_p = pd.read_csv(player_file)
        df_players_list.append(df_p)

df_players = pd.concat(df_players_list, ignore_index=True)

# ---> ΚΑΘΑΡΙΣΜΟΣ
df_players['player_name'] = df_players['player_name'].apply(clean_name)
df_understat_shots['player_assisted'] = df_understat_shots['player_assisted'].apply(clean_name)

# Σιγουρεύουμε ότι οι χρονιές είναι integers
df_understat_shots['season'] = df_understat_shots['season'].fillna(-1).astype(int)
df_players['year'] = df_players['year'].fillna(-1).astype(int)

# 2. ΤΟ FIX: Κρατάμε ΑΥΣΤΗΡΑ μόνο μία εγγραφή ανά παίκτη και ανά χρονιά!
# Αν έχει 2 ομάδες, κρατάμε την πρώτη εμφάνιση (το ID είναι ίδιο έτσι κι αλλιώς).
df_players_lookup = df_players[['player_name', 'year', 'id']].drop_duplicates(subset=['player_name', 'year'], keep='first')

# Διαγράφουμε την αρχική κενή στήλη (αν υπάρχει) για να την πάρουμε καθαρή από το merge
if 'uid_assisted' in df_understat_shots.columns:
    df_understat_shots = df_understat_shots.drop(columns=['uid_assisted'])

# 3. Κάνουμε το merge ΠΑΝΩ στο ίδιο dataframe για να προστατέψουμε το index
df_understat_shots = df_understat_shots.merge(
    df_players_lookup,
    left_on=['player_assisted', 'season'],
    right_on=['player_name', 'year'],
    how='left'
)

# Μετονομάζουμε το 'id' σε 'uid_assisted' και πετάμε τα σκουπίδια
df_understat_shots = df_understat_shots.rename(columns={'id': 'uid_assisted'})
df_understat_shots = df_understat_shots.drop(columns=['player_name', 'year'])

# 4. Υπολογισμοί και Στατιστικά
mask_has_assist = df_understat_shots['player_assisted'].notna() & (df_understat_shots['player_assisted'] != '')
df_with_assist = df_understat_shots[mask_has_assist]

shots_matched = df_with_assist['uid_assisted'].notna().sum()
shots_unmatched = df_with_assist['uid_assisted'].isna().sum()

unique_matched_players = df_with_assist[df_with_assist['uid_assisted'].notna()]['player_assisted'].nunique()
unique_unmatched_players = df_with_assist[df_with_assist['uid_assisted'].isna()]['player_assisted'].unique()

print(f"Συνολικά σουτ που προήλθαν από ασίστ: {len(df_with_assist)}")
print(f"--> Βρέθηκε και προστέθηκε uid_assisted σε: {shots_matched} σουτ")
print(f"--> ΔΕΝ βρέθηκε match για: {shots_unmatched} σουτ (αφορά {len(unique_unmatched_players)} μοναδικούς παίκτες)")

if len(unique_unmatched_players) > 0:
    print("\nΠαίκτες που πραγματικά λείπουν (εμφάνιση πρώτων 20):")
    for player in unique_unmatched_players[:20]:
        print(f" - {player}")

Συνολικά σουτ που προήλθαν από ασίστ: 451660
--> Βρέθηκε και προστέθηκε uid_assisted σε: 451660 σουτ
--> ΔΕΝ βρέθηκε match για: 0 σουτ (αφορά 0 μοναδικούς παίκτες)


Ελεγχος για NotNULL (restriction του table της DB) και μετατροπη DataTypes στο DataFrame

In [22]:
# 1. Μετατροπή των στηλών που πρέπει να είναι ακέραιοι (integers)
# Χρησιμοποιούμε το 'Int64' (nullable integer) για να μην σκάσει στα NaN
int_columns = ['shot_id', 'match_id', 'uid_understat', 'minute', 'season', 'is_goal', 'uid_assisted']

for col in int_columns:
    df_understat_shots[col] = df_understat_shots[col].astype('Int64')

# Οι συντεταγμένες και το xG παραμένουν float (double precision)
float_columns = ['x_loc', 'y_loc', 'understat_xg']
for col in float_columns:
    df_understat_shots[col] = df_understat_shots[col].astype(float)


# 2. Ορισμός της ακριβούς σειράς των στηλών βάσει του SQL Schema
# (παραλείπουμε τα distance_to_goal και angle_to_goal γιατί παράγονται αυτόματα - GENERATED ALWAYS - στην SQL)
final_columns = [
    'shot_id', 'match_id', 'uid_understat',
    'minute', 'season', 'league', 'h_a',
    'x_loc', 'y_loc',
    'play_pattern', 'last_action', 'player_assisted', 'uid_assisted',
    'body_part', 'outcome', 'is_goal', 'understat_xg'
]

# Κρατάμε μόνο αυτές τις στήλες
df_understat_shots = df_understat_shots[final_columns]


# 3. Έλεγχος για NOT NULL πεδία
not_null_cols = ['shot_id', 'match_id', 'uid_understat']
print("--- ΕΛΕΓΧΟΣ ΓΙΑ NOT NULL ΠΕΔΙΑ ---")
for col in not_null_cols:
    missing = df_understat_shots[col].isna().sum()
    if missing > 0:
        print(f"❌ ΠΡΟΣΟΧΗ: Η στήλη {col} έχει {missing} κενές τιμές!")
    else:
        print(f"✅ Η στήλη {col} είναι πλήρης (0 κενά).")

print("\n--- ΤΕΛΙΚΟΙ ΤΥΠΟΙ ΔΕΔΟΜΕΝΩΝ (Data Types) ---")
print(df_understat_shots.dtypes)

# Αντικαθιστούμε τα np.nan στα strings με None, το οποίο μεταφράζεται σωστά σε SQL NULL
# (προαιρετικό, αλλά βοηθάει πολύ τις βιβλιοθήκες SQL)
df_understat_shots = df_understat_shots.where(pd.notnull(df_understat_shots), None)

--- ΕΛΕΓΧΟΣ ΓΙΑ NOT NULL ΠΕΔΙΑ ---
✅ Η στήλη shot_id είναι πλήρης (0 κενά).
✅ Η στήλη match_id είναι πλήρης (0 κενά).
✅ Η στήλη uid_understat είναι πλήρης (0 κενά).

--- ΤΕΛΙΚΟΙ ΤΥΠΟΙ ΔΕΔΟΜΕΝΩΝ (Data Types) ---
shot_id              Int64
match_id             Int64
uid_understat        Int64
minute               Int64
season               Int64
league                 str
h_a                    str
x_loc              float64
y_loc              float64
play_pattern           str
last_action            str
player_assisted        str
uid_assisted         Int64
body_part              str
outcome                str
is_goal              Int64
understat_xg       float64
dtype: object


Fix X, Y to 0-1 scale

In [23]:
# --- ΔΙΟΡΘΩΣΗ ΣΥΝΤΕΤΑΓΜΕΝΩΝ ---
def fix_coordinates(val):
    if pd.isna(val):
        return val
    # Αν η τιμή είναι πάνω από 1 (ενώ θα έπρεπε να είναι 0 έως 1)
    if val > 1:
        # Μετράμε πόσα ψηφία έχει το ακέραιο μέρος του αριθμού (π.χ. το 435 έχει 3)
        num_digits = len(str(int(val)))
        # Διαιρούμε με το 10 εις την δύναμη των ψηφίων (10^3 = 1000)
        return val / (10 ** num_digits)
    return val

print("Διορθώνονται οι συντεταγμένες x, y...")
df_understat_shots['x_loc'] = df_understat_shots['x_loc'].apply(fix_coordinates)
df_understat_shots['y_loc'] = df_understat_shots['y_loc'].apply(fix_coordinates)


# --- ΕΛΕΓΧΟΣ ΚΑΙ ΚΛΕΙΔΩΜΑ ΤΥΠΩΝ (DATA TYPES) ---
# 1. Μετατροπή των στηλών που πρέπει να είναι ακέραιοι (integers)
int_columns = ['shot_id', 'match_id', 'uid_understat', 'minute', 'season', 'is_goal', 'uid_assisted']

for col in int_columns:
    df_understat_shots[col] = df_understat_shots[col].astype('Int64')

# Οι συντεταγμένες και το xG παραμένουν float (double precision)
float_columns = ['x_loc', 'y_loc', 'understat_xg']
for col in float_columns:
    df_understat_shots[col] = df_understat_shots[col].astype(float)

# 2. Ορισμός της ακριβούς σειράς των στηλών βάσει του SQL Schema
final_columns = [
    'shot_id', 'match_id', 'uid_understat',
    'minute', 'season', 'league', 'h_a',
    'x_loc', 'y_loc',
    'play_pattern', 'last_action', 'player_assisted', 'uid_assisted',
    'body_part', 'outcome', 'is_goal', 'understat_xg'
]

# Κρατάμε μόνο αυτές τις στήλες
df_understat_shots = df_understat_shots[final_columns]

# 3. Έλεγχος για NOT NULL πεδία
not_null_cols = ['shot_id', 'match_id', 'uid_understat']
print("\n--- ΕΛΕΓΧΟΣ ΓΙΑ NOT NULL ΠΕΔΙΑ ---")
for col in not_null_cols:
    missing = df_understat_shots[col].isna().sum()
    if missing > 0:
        print(f"❌ ΠΡΟΣΟΧΗ: Η στήλη {col} έχει {missing} κενές τιμές!")
    else:
        print(f"✅ Η στήλη {col} είναι πλήρης (0 κενά).")

# 4. Μετατροπή των NaNs σε καθαρό None (για να μπουν ως SQL NULL)
# Προσοχή: Επειδή έχουμε Int64 (το οποίο δεν υποστηρίζει πάντα None αλλά pd.NA),
# η SQLAlchemy το διαχειρίζεται αυτόματα. Για τα string πεδία όμως βοηθάει:
# 4. Μετατροπή των NaNs σε καθαρό None στις στήλες κειμένου (για να μπουν ως SQL NULL)
string_columns = ['league', 'h_a', 'play_pattern', 'last_action', 'player_assisted', 'body_part', 'outcome']

for col in string_columns:
    df_understat_shots[col] = df_understat_shots[col].where(pd.notnull(df_understat_shots[col]), None)

print("\n--- ΤΕΛΙΚΟΙ ΤΥΠΟΙ ΔΕΔΟΜΕΝΩΝ (Data Types) ---")
print(df_understat_shots.dtypes)

# Δείγμα από τα δεδομένα μας για επιβεβαίωση
display(df_understat_shots.head())

Διορθώνονται οι συντεταγμένες x, y...

--- ΕΛΕΓΧΟΣ ΓΙΑ NOT NULL ΠΕΔΙΑ ---
✅ Η στήλη shot_id είναι πλήρης (0 κενά).
✅ Η στήλη match_id είναι πλήρης (0 κενά).
✅ Η στήλη uid_understat είναι πλήρης (0 κενά).

--- ΤΕΛΙΚΟΙ ΤΥΠΟΙ ΔΕΔΟΜΕΝΩΝ (Data Types) ---
shot_id              Int64
match_id             Int64
uid_understat        Int64
minute               Int64
season               Int64
league                 str
h_a                    str
x_loc              float64
y_loc              float64
play_pattern           str
last_action            str
player_assisted        str
uid_assisted         Int64
body_part              str
outcome                str
is_goal              Int64
understat_xg       float64
dtype: object


,shot_id,match_id,uid_understat,minute,season,league,h_a,x_loc,y_loc,play_pattern,last_action,player_assisted,uid_assisted,body_part,outcome,is_goal,understat_xg
0,24339,5447,224,11,2014,Bundesliga,h,0.903,0.239,OpenPlay,Chipped,Philipp Lahm,218,RightFoot,SavedShot,0,0.034118
1,24340,5447,392,17,2014,Bundesliga,h,0.852,0.277,OpenPlay,Pass,Philipp Lahm,218,LeftFoot,SavedShot,0,0.030941
2,24342,5447,392,26,2014,Bundesliga,h,0.803,0.277,OpenPlay,Chipped,Holger Badstuber,219,LeftFoot,BlockedShot,0,0.021718
3,24343,5447,224,28,2014,Bundesliga,h,0.871,0.324,OpenPlay,NaN,NaN,<NA>,LeftFoot,SavedShot,0,0.050345
4,24344,5447,227,29,2014,Bundesliga,h,0.918,0.531,OpenPlay,Chipped,Gianluca Gaudino,416,RightFoot,SavedShot,0,0.111078


CSV Export

In [11]:
# Εξαγωγή του τελικού DataFrame σε αρχείο CSV
export_filename = "understat_shots_final.csv"

df_understat_shots.to_csv(export_filename, index=False, encoding='utf-8-sig')

print(f"✅ Η εξαγωγή ολοκληρώθηκε! Το αρχείο αποθηκεύτηκε ως '{export_filename}' στον φάκελο του project σου.")

✅ Η εξαγωγή ολοκληρώθηκε! Το αρχείο αποθηκεύτηκε ως 'understat_shots_final.csv' στον φάκελο του project σου.


Confirmation x,y is float and not Integer. and player_assisted + uid_assisted is None, NaN, Empty

In [24]:
# --- ΕΛΕΓΧΟΣ 1: x_loc και y_loc ---
print("--- ΕΛΕΓΧΟΣ ΣΥΝΤΕΤΑΓΜΕΝΩΝ (x_loc, y_loc) ---")

# 1α. Έλεγχος αν υπάρχουν τιμές εκτός των λογικών ορίων (0 έως 1)
invalid_x = df_understat_shots[(df_understat_shots['x_loc'] < 0) | (df_understat_shots['x_loc'] > 1)]
invalid_y = df_understat_shots[(df_understat_shots['y_loc'] < 0) | (df_understat_shots['y_loc'] > 1)]

if len(invalid_x) == 0 and len(invalid_y) == 0:
    print("✅ Όλες οι συντεταγμένες x και y είναι αυστηρά μέσα στο εύρος 0 έως 1.")
else:
    print(f"❌ ΠΡΟΣΟΧΗ: Βρέθηκαν {len(invalid_x)} τιμές x_loc και {len(invalid_y)} τιμές y_loc εκτός του (0-1).")

# 1β. Έλεγχος για το αν υπάρχουν τιμές που λειτουργούν αυστηρά ως "ακέραιοι" (χωρίς δεκαδικό μέρος π.χ. 0.0 ή 1.0)
def is_strictly_integer(val):
    if pd.isna(val):
        return False
    return val.is_integer()

integer_x = df_understat_shots[df_understat_shots['x_loc'].apply(is_strictly_integer)]
integer_y = df_understat_shots[df_understat_shots['y_loc'].apply(is_strictly_integer)]

if len(integer_x) > 0 or len(integer_y) > 0:
    print(f"ℹ️ Βρέθηκαν {len(integer_x)} τιμές στο x_loc και {len(integer_y)} στο y_loc που έχουν μηδενικό δεκαδικό (π.χ. 0.0 ή 1.0).")
else:
    print("✅ Δεν βρέθηκαν καθόλου ακέραιες τιμές στα x, y. Όλες έχουν δεκαδικό μέρος.")


print("\n--- ΕΛΕΓΧΟΣ ASSISTED PLAYER ---")
# --- ΕΛΕΓΧΟΣ 2: Συνέπεια μεταξύ player_assisted και uid_assisted ---

# Βρίσκουμε τις γραμμές όπου ΔΕΝ υπάρχει assister (είναι None ή NaN ή άδειο string)
mask_no_assister = df_understat_shots['player_assisted'].isna() | (df_understat_shots['player_assisted'] == '')

# Ελέγχουμε αν σε κάποια από αυτές τις γραμμές, το uid_assisted ΔΕΝ είναι κενό
inconsistent_assists = df_understat_shots[mask_no_assister & df_understat_shots['uid_assisted'].notna()]

if len(inconsistent_assists) == 0:
    print("✅ ΕΠΙΒΕΒΑΙΩΣΗ: Όταν το όνομα του assister είναι NULL, τότε και το ID (uid_assisted) είναι NULL!")
else:
    print(f"❌ ΠΡΟΣΟΧΗ: Βρέθηκαν {len(inconsistent_assists)} εγγραφές όπου λείπει το όνομα, αλλά υπάρχει ID!")
    display(inconsistent_assists[['shot_id', 'player_assisted', 'uid_assisted']].head())

--- ΕΛΕΓΧΟΣ ΣΥΝΤΕΤΑΓΜΕΝΩΝ (x_loc, y_loc) ---
✅ Όλες οι συντεταγμένες x και y είναι αυστηρά μέσα στο εύρος 0 έως 1.
ℹ️ Βρέθηκαν 4 τιμές στο x_loc και 5 στο y_loc που έχουν μηδενικό δεκαδικό (π.χ. 0.0 ή 1.0).

--- ΕΛΕΓΧΟΣ ASSISTED PLAYER ---
✅ ΕΠΙΒΕΒΑΙΩΣΗ: Όταν το όνομα του assister είναι NULL, τότε και το ID (uid_assisted) είναι NULL!


Database Connection

In [25]:
import os
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from sqlalchemy.exc import SQLAlchemyError

load_dotenv(find_dotenv())

def connect_to_db():
    host = os.getenv("DB_HOST", "localhost")
    port = os.getenv("DB_PORT", "5432")
    database = os.getenv("DB_NAME")
    user = os.getenv("DB_USER")
    password = os.getenv("DB_PASSWORD")

    if not all([database, user, password]):
        print("Σφάλμα: Δεν βρέθηκαν οι απαραίτητες μεταβλητές στο .env!")
        return None

    try:
        connection_uri = f"postgresql://{user}:{password}@{host}:{port}/{database}"
        engine = create_engine(connection_uri)
        with engine.connect() as connection:
            print(f"Επιτυχής σύνδεση στη βάση '{database}' στο host '{host}'!")
        return engine

    except SQLAlchemyError as e:
        print(f"Σφάλμα κατά τη σύνδεση στη ΒΔ: {e}")
        return None


print("Εκκίνηση σύνδεσης με τη βάση...")
engine = connect_to_db()

if engine is None:
    print("Τερματισμός προγράμματος λόγω αποτυχίας σύνδεσης.")
    exit(1)

Εκκίνηση σύνδεσης με τη βάση...
Επιτυχής σύνδεση στη βάση 'thesis_db' στο host 'dell-micro'!


Insert understat_id & Player name to table "understat_player_info"

In [26]:
import time

print("⏳ Ετοιμασία του λεξικού παικτών (understat_player_info)...")

# 1. Απομονώνουμε το ID και το Όνομα, και κρατάμε ΜΟΝΟ τις μοναδικές εγγραφές βάσει του ID
df_players_db = df_players[['id', 'player_name']].drop_duplicates(subset=['id']).copy()

# Μετονομάζουμε το 'id' σε 'uid_understat' για να ταιριάζει με τη βάση
df_players_db = df_players_db.rename(columns={'id': 'uid_understat'})

# Βεβαιωνόμαστε ότι τα ονόματα είναι καθαρά (τρέχουμε τη συνάρτηση clean_name που φτιάξαμε πριν)
df_players_db['player_name'] = df_players_db['player_name'].apply(clean_name)

print(f"Βρέθηκαν {len(df_players_db)} μοναδικοί παίκτες. Ξεκινάει το INSERT...")

# 2. Εισαγωγή στη βάση
start_time = time.time()
table_players = 'understat_player_info'

try:
    with engine.begin() as conn:
        df_players_db.to_sql(
            name=table_players,
            con=conn,
            if_exists='append',
            index=False,
            method='multi'
        )
    elapsed = round(time.time() - start_time, 2)
    print(f"✅ ΕΠΙΤΥΧΙΑ! Μπήκαν όλοι οι παίκτες στον πίνακα {table_players} σε {elapsed} δευτερόλεπτα.")
except Exception as e:
    print(f"❌ Σφάλμα κατά την εισαγωγή των παικτών: {e}")

⏳ Ετοιμασία του λεξικού παικτών (understat_player_info)...
Βρέθηκαν 11277 μοναδικοί παίκτες. Ξεκινάει το INSERT...
❌ Σφάλμα κατά την εισαγωγή των παικτών: Execution failed on sql 'INSERT INTO understat_player_info (uid_understat, player_name) VALUES (:uid_understat_m0, :player_name_m0), (:uid_understat_m1, :player_name_m1), (:uid_understat_m2, :player_name_m2), (:uid_understat_m3, :player_name_m3), (:uid_understat_m4, :player_name_m4), (:uid_understat_m5, :player_name_m5), (:uid_understat_m6, :player_name_m6), (:uid_understat_m7, :player_name_m7), (:uid_understat_m8, :player_name_m8), (:uid_understat_m9, :player_name_m9), (:uid_understat_m10, :player_name_m10), (:uid_understat_m11, :player_name_m11), (:uid_understat_m12, :player_name_m12), (:uid_understat_m13, :player_name_m13), (:uid_understat_m14, :player_name_m14), (:uid_understat_m15, :player_name_m15), (:uid_understat_m16, :player_name_m16), (:uid_understat_m17, :player_name_m17), (:uid_understat_m18, :player_name_m18), (:uid_unde

Insert Shots to Database

In [27]:
table_name = 'understat_shots'
print(f"⏳ Ξεκινάει η εισαγωγή {len(df_understat_shots)} εγγραφών στον πίνακα '{table_name}'...")
start_time = time.time()

try:
    # Το engine.begin() ανοίγει ένα transaction που κάνει ΑΥΤΟΜΑΤΑ commit
    # αν δεν υπάρξει σφάλμα, ή rollback αν "σκάσει" κάτι.
    with engine.begin() as connection:
        df_understat_shots.to_sql(
            name=table_name,
            con=connection,
            if_exists='append',
            index=False,
            chunksize=10000,
            method='multi'
        )

    elapsed_time = round(time.time() - start_time, 2)
    print(f"✅ ΕΠΙΤΥΧΙΑ! Το transaction έγινε commit. Χρόνος: {elapsed_time} δευτερόλεπτα.")

except Exception as e:
    print(f"❌ Σφάλμα κατά την εισαγωγή: {e}")

⏳ Ξεκινάει η εισαγωγή 615225 εγγραφών στον πίνακα 'understat_shots'...
✅ ΕΠΙΤΥΧΙΑ! Το transaction έγινε commit. Χρόνος: 294.52 δευτερόλεπτα.


Import Match Info to dataframe

In [29]:
base_path = '../dataset-understat'
all_match_info = []

print("⏳ Συλλογή δεδομένων αγώνων (match_info.csv) από όλα τα πρωταθλήματα...")

# 1. Διασχίζουμε τους υποφακέλους των πρωταθλημάτων
for league_folder in os.listdir(base_path):
    league_path = os.path.join(base_path, league_folder)

    if os.path.isdir(league_path):
        match_info_file = os.path.join(league_path, 'match_info.csv')

        if os.path.exists(match_info_file):
            # Προσοχή στο sep=';' που χρησιμοποιεί το understat
            df_temp = pd.read_csv(match_info_file, sep=';')
            all_match_info.append(df_temp)
            print(f"  Ενσωματώθηκε: {league_folder} ({len(df_temp)} αγώνες)")

# 2. Ενώνουμε όλα τα μικρά dataframes σε ένα μεγάλο
df_us_matches = pd.concat(all_match_info, ignore_index=True)
print(f"Συνολικά βρέθηκαν {len(df_us_matches)} αγώνες.")

# 3. Φιλτράρισμα & Μετονομασία στηλών
cols_to_keep = {
    'id': 'match_id',
    'date': 'match_date',
    'league': 'league',
    'season': 'season',
    'team_h': 'home_team',
    'team_a': 'away_team',
    'h_goals': 'home_goals',
    'a_goals': 'away_goals',
    'h_xg': 'home_xg',
    'a_xg': 'away_xg',
    'h_deep': 'home_deep',
    'a_deep': 'away_deep',
    'h_ppda': 'home_ppda',
    'a_ppda': 'away_ppda'
}

df_us_matches = df_us_matches[list(cols_to_keep.keys())].rename(columns=cols_to_keep)

# 4. Μετατροπή ημερομηνίας σε σωστό τύπο DateTime για τη βάση
df_us_matches['match_date'] = pd.to_datetime(df_us_matches['match_date'])

# 5. Μετατροπή των ID και των Goals σε Int64 (για αποφυγή float λαθών)
int_cols = ['match_id', 'season', 'home_goals', 'away_goals']
for col in int_cols:
    df_us_matches[col] = df_us_matches[col].astype('Int64')

print(f"\n✅ Το DataFrame df_us_matches είναι έτοιμο!")
display(df_us_matches.head())

⏳ Συλλογή δεδομένων αγώνων (match_info.csv) από όλα τα πρωταθλήματα...
  Ενσωματώθηκε: Bundesliga (3672 αγώνες)
  Ενσωματώθηκε: EPL (4560 αγώνες)
  Ενσωματώθηκε: La_Liga (4560 αγώνες)
  Ενσωματώθηκε: Ligue_1 (4237 αγώνες)
  Ενσωματώθηκε: RFPL (2880 αγώνες)
  Ενσωματώθηκε: Serie_A (4560 αγώνες)
Συνολικά βρέθηκαν 24469 αγώνες.

✅ Το DataFrame df_us_matches είναι έτοιμο!


,match_id,match_date,league,season,home_team,away_team,home_goals,away_goals,home_xg,away_xg,home_deep,away_deep,home_ppda,away_ppda
0,5447,2014-08-22 19:30:00,Bundesliga,2014,Bayern Munich,Wolfsburg,2,1,2.57012,1.19842,5,4,9.6250,21.8500
1,5448,2014-08-23 14:30:00,Bundesliga,2014,Hoffenheim,Augsburg,2,0,1.52873,0.28078,6,2,6.5882,5.7391
2,5449,2014-08-23 14:30:00,Bundesliga,2014,Hannover 96,Schalke 04,2,1,1.17979,0.95666,4,3,9.6429,6.0556
3,5450,2014-08-23 14:30:00,Bundesliga,2014,Hertha Berlin,Werder Bremen,2,2,1.75585,1.19453,3,5,5.6857,9.8696
4,5451,2014-08-23 14:30:00,Bundesliga,2014,Eintracht Frankfurt,Freiburg,1,0,1.75331,1.38084,3,13,10.5172,3.4651


Import to DB Table "understat_matches"

In [31]:
table_name = 'understat_matches'
print(f"⏳ Δημιουργία πίνακα (αν δεν υπάρχει) και εισαγωγή {len(df_us_matches)} αγώνων...")

try:
    with engine.begin() as conn:

        # Εισαγωγή των δεδομένων του DataFrame
        df_us_matches.to_sql(
            name=table_name,
            con=conn,
            if_exists='append',
            index=False,
            chunksize=10000,
            method='multi'
        )

    print(f"✅ ΕΠΙΤΥΧΙΑ!")

except Exception as e:
    print(f"❌ Σφάλμα κατά τη δημιουργία/εισαγωγή: {e}")

⏳ Δημιουργία πίνακα (αν δεν υπάρχει) και εισαγωγή 24469 αγώνων...
✅ ΕΠΙΤΥΧΙΑ!
